In [1]:
import sys
import os

# So notebooks can access src/
sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
from datetime import datetime

In [3]:
products = pd.read_parquet("../data/processed/products_clean.parquet")
behavior = pd.read_csv("../data/raw/user_behavior.csv")

behavior['timestamp'] = pd.to_datetime(behavior['timestamp'])

In [4]:
def price_category(price):
    if price < 10000:
        return "budget"
    elif price < 30000:
        return "mid"
    else:
        return "premium"

products['price_category'] = products['price'].apply(price_category)


def room_type(category):
    if category == 'sofa':
        return 'living_room'
    elif category in ['bed', 'wardrobe']:
        return 'bedroom'
    else:
        return 'office'

products['room_type'] = products['category'].apply(room_type)

products.head()

,product_id,name,category,material,color,style,price,rating,description,category_encoded,material_encoded,style_encoded,price_category,room_type
0,0,modern sofa,sofa,Glass,White,Contemporary,7837.36,2.8,Contemporary White Glass Sofa,2,1,0,budget,living_room
1,1,modern wardrobe,wardrobe,Leather,Brown,Modern,2460.06,3.0,Modern Brown Leather Wardrobe,4,2,3,budget,bedroom
2,2,industrial wardrobe,wardrobe,Wood,Beige,Contemporary,36084.96,4.3,Contemporary Beige Wood Wardrobe,4,4,0,premium,bedroom
3,3,contemporary bed,bed,Fabric,Beige,Vintage,40662.09,2.5,Vintage Beige Fabric Bed,0,0,4,premium,bedroom
4,4,minimalist chair,chair,Glass,Grey,Contemporary,11550.37,4.4,Contemporary Grey Glass Chair,1,1,0,mid,office


In [5]:
purchases = behavior[behavior['event_type'] == 'purchase']

now = datetime.now()

rfm = purchases.groupby('user_id').agg({
    'timestamp': lambda x: (now - x.max()).days,
    'product_id': 'count'
}).rename(columns={
    'timestamp': 'recency',
    'product_id': 'frequency'
})

merged = purchases.merge(products, on='product_id')
monetary = merged.groupby('user_id')['price'].sum()

rfm['monetary'] = monetary
rfm = rfm.fillna(0)

rfm.head()

,recency,frequency,monetary
user_id,,,
0,118,3,39305.92
1,138,1,13208.24
2,9,3,72032.04
3,12,3,117136.82
6,140,2,24783.60
